In [1]:
!pip install scikit-learn pandas numpy

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import pickle


In [3]:
df = pd.read_csv("cleaned_data.csv")
df.head()


,Skills,Experience (Years),Education,Certifications,Job Role,Projects Count,strength
0,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,8,Strong
1,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,1,Strong
2,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,7,Average
3,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,0,Strong
4,"SQL, React, Java",4,PhD,NaN,Software Engineer,9,Strong


In [5]:
df.columns


Index(['Skills', 'Experience (Years)', 'Education', 'Certifications',
       'Job Role', 'Projects Count', 'strength', 'combined_text'],
      dtype='object')

In [6]:
df.rename(columns={'strength': 'Strength'}, inplace=True)

In [7]:
df['combined_text'] = (
    df['Skills'].astype(str) + ' ' +
    df['Education'].astype(str) + ' ' +
    df['Certifications'].astype(str) + ' ' +
    df['Job Role'].astype(str) + ' ' +
    df['Experience (Years)'].astype(str) + ' ' +
    df['Projects Count'].astype(str)
)

df[['combined_text', 'Strength']].head()


,combined_text,Strength
0,"TensorFlow, NLP, Pytorch B.Sc nan AI Researche...",Strong
1,"Deep Learning, Machine Learning, Python, SQL M...",Strong
2,"Ethical Hacking, Cybersecurity, Linux MBA Deep...",Average
3,"Python, Pytorch, TensorFlow B.Tech AWS Certifi...",Strong
4,"SQL, React, Java PhD nan Software Engineer 4 9",Strong


In [8]:
X = df['combined_text']
y = df['Strength']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [9]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [12]:
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train_tfidf, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000)

In [13]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.435

Classification Report:

              precision    recall  f1-score   support

     Average       0.31      0.62      0.41        39
      Strong       0.83      0.39      0.53       139
        Weak       0.16      0.41      0.23        22

    accuracy                           0.43       200
   macro avg       0.43      0.47      0.39       200
weighted avg       0.65      0.43      0.47       200



In [14]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)


MultinomialNB()

In [15]:
y_pred_nb = nb_model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))


Accuracy: 0.695

Classification Report:

              precision    recall  f1-score   support

     Average       0.00      0.00      0.00        39
      Strong       0.69      1.00      0.82       139
        Weak       0.00      0.00      0.00        22

    accuracy                           0.69       200
   macro avg       0.23      0.33      0.27       200
weighted avg       0.48      0.69      0.57       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Model Selection Summary

- Logistic Regression: High accuracy but biased toward majority class
- Naive Bayes: Similar bias observed
- Linear SVM: Best balance across classes → selected final model



In [16]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(class_weight='balanced')
svm_model.fit(X_train_tfidf, y_train)


LinearSVC(class_weight='balanced')

In [17]:
y_pred_svm = svm_model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))


Accuracy: 0.625

Classification Report:

              precision    recall  f1-score   support

     Average       0.29      0.10      0.15        39
      Strong       0.71      0.86      0.78       139
        Weak       0.11      0.09      0.10        22

    accuracy                           0.62       200
   macro avg       0.37      0.35      0.34       200
weighted avg       0.56      0.62      0.58       200



In [18]:
with open("resume_model.pkl", "wb") as f:
    pickle.dump(svm_model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
